# SQL Interview Questions

This notebook contains 10 important SQL interview questions covering basic queries, aggregate functions, JOINs, subqueries, conditional statements, window functions, and CTEs.

Each question includes the problem statement, SQL query, expected output, and explanation.

**Environment:** Google Colab, SQLite, and Python.

In [1]:

import sqlite3
import pandas as pd

# Create an in-memory database
connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

# Create Employees table
cursor.execute("""
CREATE TABLE Employees (
    employee_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    department_id INTEGER,
    salary INTEGER,
    city TEXT,
    hire_date TEXT,
    manager_id INTEGER,
    FOREIGN KEY (department_id)
        REFERENCES Departments(department_id)
)
""")

# Create Departments table
cursor.execute("""
CREATE TABLE Departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL,
    location TEXT
)
""")

# Insert department records
departments = [
    (1, "IT", "Bengaluru"),
    (2, "HR", "Mysuru"),
    (3, "Finance", "Hassan"),
    (4, "Marketing", "Mangaluru")
]

cursor.executemany("""
INSERT INTO Departments
(department_id, department_name, location)
VALUES (?, ?, ?)
""", departments)

# Insert employee records
employees = [
    (1, "Asha", 1, 60000, "Bengaluru", "2024-01-15", 3),
    (2, "Ravi", 2, 45000, "Mysuru", "2023-06-10", 3),
    (3, "Priya", 1, 75000, "Bengaluru", "2022-03-20", None),
    (4, "Kiran", 3, 50000, "Hassan", "2024-08-05", 7),
    (5, "Neha", 2, 55000, "Bengaluru", "2023-11-12", 2),
    (6, "Arjun", 1, 65000, "Mysuru", "2022-09-01", 3),
    (7, "Meena", 3, 70000, "Hassan", "2021-05-18", None),
    (8, "Vijay", 1, 50000, "Bengaluru", "2025-02-01", 3)
]

cursor.executemany("""
INSERT INTO Employees
(employee_id, name, department_id, salary, city, hire_date, manager_id)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", employees)

connection.commit()

print("Database and tables created successfully.")


Database and tables created successfully.


In [2]:

pd.read_sql_query("SELECT * FROM Employees", connection)


,employee_id,name,department_id,salary,city,hire_date,manager_id
0,1,Asha,1,60000,Bengaluru,2024-01-15,3.0
1,2,Ravi,2,45000,Mysuru,2023-06-10,3.0
2,3,Priya,1,75000,Bengaluru,2022-03-20,NaN
3,4,Kiran,3,50000,Hassan,2024-08-05,7.0
4,5,Neha,2,55000,Bengaluru,2023-11-12,2.0
5,6,Arjun,1,65000,Mysuru,2022-09-01,3.0
6,7,Meena,3,70000,Hassan,2021-05-18,NaN
7,8,Vijay,1,50000,Bengaluru,2025-02-01,3.0


In [3]:

pd.read_sql_query("SELECT * FROM Departments", connection)


,department_id,department_name,location
0,1,IT,Bengaluru
1,2,HR,Mysuru
2,3,Finance,Hassan
3,4,Marketing,Mangaluru


## Question 1: Retrieve and Sort Employee Records

**Interview prompt:**

Write an SQL query to display the names, salaries, and cities of employees earning more than 50,000. Display the highest-paid employees first.

**Concepts:** `SELECT`, `WHERE`, `ORDER BY`.

In [5]:

query = """
SELECT name, salary, city
FROM Employees
WHERE salary > 50000
ORDER BY salary DESC;
"""

pd.read_sql_query(query, connection)


,name,salary,city
0,Priya,75000,Bengaluru
1,Meena,70000,Hassan
2,Arjun,65000,Mysuru
3,Asha,60000,Bengaluru
4,Neha,55000,Bengaluru


**Explanation:**

- `SELECT` specifies the columns to display.
- `FROM Employees` identifies the table.
- `WHERE salary > 50000` filters employees earning more than 50,000.
- `ORDER BY salary DESC` sorts the results from highest to lowest salary.

**Time complexity:** Typically O(n) for filtering, with additional sorting cost of O(k log k), where n is the number of employees and k is the number of matching rows.

## Question 2: Calculate Department-Wise Salary Statistics

**Interview prompt:**

Write an SQL query to display each department's employee count, total salary, average salary, minimum salary, and maximum salary. Display only departments with an average salary greater than 55,000.

**Concepts:** `COUNT()`, `SUM()`, `AVG()`, `MIN()`, `MAX()`, `GROUP BY`, `HAVING`.

In [15]:

query = """
SELECT
    d.department_name,
    COUNT(e.employee_id) AS employee_count,
    SUM(e.salary) AS total_salary,
    ROUND(AVG(e.salary), 2) AS average_salary,
    MIN(e.salary) AS minimum_salary,
    MAX(e.salary) AS maximum_salary
FROM Departments d
JOIN Employees e
    ON d.department_id = e.department_id
GROUP BY d.department_id, d.department_name
HAVING AVG(e.salary) > 55000
ORDER BY average_salary DESC;
"""

pd.read_sql_query(query, connection)


,department_name,employee_count,total_salary,average_salary,minimum_salary,maximum_salary
0,IT,4,250000,62500.0,50000,75000
1,Finance,2,120000,60000.0,50000,70000


- Aggregate functions calculate statistics across rows.
- `JOIN` connects employees to their departments.
- `GROUP BY` calculates statistics separately for each department.
- `HAVING` filters groups based on their average salary.
- `ORDER BY` sorts the qualifying departments by average salary.

**Important:** The IT average is 62,500, and the Finance average is 60,000. Both qualify because they exceed 55,000.

## Question 3: Display Employees with Department Details

**Interview prompt:**

Write an SQL query to display each employee's name, salary, department name, and department location. Return only employees who have a matching department.

**Concept:** `INNER JOIN`.

In [14]:

query = """
SELECT
    e.name,
    e.salary,
    d.department_name,
    d.location AS department_location
FROM Employees e
INNER JOIN Departments d
    ON e.department_id = d.department_id
ORDER BY e.name;
"""

pd.read_sql_query(query, connection)


,name,salary,department_name,department_location
0,Arjun,65000,IT,Bengaluru
1,Asha,60000,IT,Bengaluru
2,Kiran,50000,Finance,Hassan
3,Meena,70000,Finance,Hassan
4,Neha,55000,HR,Mysuru
5,Priya,75000,IT,Bengaluru
6,Ravi,45000,HR,Mysuru
7,Vijay,50000,IT,Bengaluru


- `INNER JOIN` returns rows with matching values in both tables.
- `e` and `d` are table aliases for `Employees` and `Departments`.
- The `ON` condition matches each employee's department ID to the corresponding department ID.

**Interview tip:** Use `INNER JOIN` when you only want matching records from both tables.

## Question 4: Find Departments Without Employees

**Interview prompt:**

Display all departments, including departments that have no employees. Show the department name and the number of employees in each department.

**Concepts:** `LEFT JOIN`, `COUNT()`, `GROUP BY`, `NULL` handling.

In [13]:

query = """
SELECT
    d.department_name,
    COUNT(e.employee_id) AS employee_count
FROM Departments d
LEFT JOIN Employees e
    ON d.department_id = e.department_id
GROUP BY d.department_id, d.department_name
ORDER BY d.department_id;
"""

pd.read_sql_query(query, connection)


,department_name,employee_count
0,IT,4
1,HR,2
2,Finance,2
3,Marketing,0


- `LEFT JOIN` keeps every row from the left table, even if no match exists in the right table.
- Unmatched employee columns become `NULL`.
- `COUNT(e.employee_id)` returns 0 for Marketing because there is no matching employee ID.
- Using `COUNT(*)` instead would count the unmatched department row, giving 1 for Marketing. This is a common interview trap.

## Question 5: Practise Different Types of JOINs

**Interview prompt:**

Write queries demonstrating:
1. `RIGHT JOIN` to keep all departments.
2. `FULL OUTER JOIN` to keep all employees and departments.
3. `CROSS JOIN` to generate combinations.
4. A self JOIN to display each employee with their manager.

**Concepts:** Different JOIN types and table aliases.

In [12]:

query = """
SELECT
    d.department_name,
    e.name AS employee_name
FROM Departments d
LEFT JOIN Employees e
    ON d.department_id = e.department_id
ORDER BY d.department_id, e.name;
"""

pd.read_sql_query(query, connection)


,department_name,employee_name
0,IT,Arjun
1,IT,Asha
2,IT,Priya
3,IT,Vijay
4,HR,Neha
5,HR,Ravi
6,Finance,Kiran
7,Finance,Meena
8,Marketing,None


In [11]:

query = """
SELECT
    e.name AS employee_name,
    d.department_name
FROM Employees e
LEFT JOIN Departments d
    ON e.department_id = d.department_id

UNION ALL

SELECT
    e.name AS employee_name,
    d.department_name
FROM Departments d
LEFT JOIN Employees e
    ON d.department_id = e.department_id
WHERE e.employee_id IS NULL;
"""

pd.read_sql_query(query, connection)


,employee_name,department_name
0,Asha,IT
1,Ravi,HR
2,Priya,IT
3,Kiran,Finance
4,Neha,HR
5,Arjun,IT
6,Meena,Finance
7,Vijay,IT
8,None,Marketing


In [10]:

query = """
SELECT e.name, d.department_name
FROM Employees e
CROSS JOIN Departments d
LIMIT 10;
"""
pd.read_sql_query(query, connection)


,name,department_name
0,Asha,IT
1,Asha,HR
2,Asha,Finance
3,Asha,Marketing
4,Ravi,IT
5,Ravi,HR
6,Ravi,Finance
7,Ravi,Marketing
8,Priya,IT
9,Priya,HR


In [9]:

query = """
SELECT
    e.name AS employee_name,
    m.name AS manager_name
FROM Employees e
LEFT JOIN Employees m
    ON e.manager_id = m.employee_id
ORDER BY e.employee_id;
"""

pd.read_sql_query(query, connection)

,employee_name,manager_name
0,Asha,Priya
1,Ravi,Priya
2,Priya,None
3,Kiran,Meena
4,Neha,Ravi
5,Arjun,Priya
6,Meena,None
7,Vijay,Priya


- `RIGHT JOIN` keeps every row from the right table.
- `FULL OUTER JOIN` keeps matched and unmatched rows from both tables.
- `CROSS JOIN` returns every possible combination of rows.
- A self JOIN joins a table to itself. Here, `e` represents the employee and `m` represents their manager.
- The full outer join query uses `LEFT JOIN` and `UNION ALL` to work on SQLite versions without native full outer join support. The second part adds only unmatched departments, avoiding duplicate matches.

## Question 6: Find Employees Earning Above Average

 prompt:

Find employees whose salary is greater than the average salary of all employees. Also practise `IN` and `EXISTS` for filtering using another query.

Concepts:Subqueries, `AVG()`, `IN`, `EXISTS`.

In [8]:

query = """
SELECT name, salary
FROM Employees
WHERE salary > (
    SELECT AVG(salary)
    FROM Employees
)
ORDER BY salary DESC;
"""

pd.read_sql_query(query, connection)


,name,salary
0,Priya,75000
1,Meena,70000
2,Arjun,65000
3,Asha,60000


In [ ]:

query = """
SELECT name, department_id
FROM Employees
WHERE department_id IN (
    SELECT department_id
    FROM Departments
    WHERE location = 'Bengaluru'
);
"""

pd.read_sql_query(query, connection)


In [7]:

query = """
SELECT d.department_name
FROM Departments d
WHERE EXISTS (
    SELECT 1
    FROM Employees e
    WHERE e.department_id = d.department_id
);
"""

pd.read_sql_query(query, connection)


,department_name
0,IT
1,HR
2,Finance


- A subquery is a query nested inside another SQL statement.
- `IN` checks whether a value matches any value returned by a subquery.
- `EXISTS` checks whether a subquery returns at least one row.
- The average salary of all 8 employees is 58,750, so Question 6A returns employees earning more than 58,750.

## Question 7: Categorize Employees by Salary

prompt:

Write an SQL query to categorize employees as:
- High Salary: salary greater than or equal to 65,000
- Medium Salary: salary between 50,000 and 64,999
- Low Salary: salary below 50,000

Display each employee's name, salary, and category.

Concept:CASE WHEN

In [6]:

query = """
SELECT
    name,
    salary,
    CASE
        WHEN salary >= 65000 THEN 'High Salary'
        WHEN salary >= 50000 THEN 'Medium Salary'
        ELSE 'Low Salary'
    END AS salary_category
FROM Employees
ORDER BY salary DESC;
"""

pd.read_sql_query(query, connection)


,name,salary,salary_category
0,Priya,75000,High Salary
1,Meena,70000,High Salary
2,Arjun,65000,High Salary
3,Asha,60000,Medium Salary
4,Neha,55000,Medium Salary
5,Kiran,50000,Medium Salary
6,Vijay,50000,Medium Salary
7,Ravi,45000,Low Salary


- `CASE` evaluates conditions in order.
- The first matching condition determines the returned value.
- `ELSE` specifies the default category if no earlier condition matches.
- `END` completes the expression.

## Question 8: Find the Second-Highest Distinct Salary
prompt:

Write an SQL query to find the second-highest distinct salary without using `LIMIT` or `TOP`.
Concepts: Subqueries, `MAX()`, distinct salary ranking.

In [17]:

query = """
SELECT MAX(salary) AS second_highest_salary
FROM Employees
WHERE salary < (
    SELECT MAX(salary)
    FROM Employees
);
"""

pd.read_sql_query(query, connection)


,second_highest_salary
0,70000


1. The inner query finds the highest salary, which is 75,000.
2. The outer query excludes salaries equal to or above 75,000.
3. `MAX()` then returns the largest remaining salary, which is 70,000.

Why this works: It finds the second-highest *distinct* salary even when multiple employees share the same salary. If no second distinct salary exists, the result is `NULL`.

## Question 9: Rank Employees Within Each Department

**prompt:**

Display each employee's name, department, salary, row number, rank, and dense rank within their department. The highest salary in each department should receive rank 1.

**Concepts:** `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()`, `PARTITION BY`.

In [16]:

query = """
SELECT
    name,
    department_id,
    salary,
    ROW_NUMBER() OVER (
        PARTITION BY department_id
        ORDER BY salary DESC, employee_id
    ) AS row_num,
    RANK() OVER (
        PARTITION BY department_id
        ORDER BY salary DESC
    ) AS salary_rank,
    DENSE_RANK() OVER (
        PARTITION BY department_id
        ORDER BY salary DESC
    ) AS dense_salary_rank
FROM Employees
ORDER BY department_id, salary DESC, employee_id;
"""

pd.read_sql_query(query, connection)


,name,department_id,salary,row_num,salary_rank,dense_salary_rank
0,Priya,1,75000,1,1,1
1,Arjun,1,65000,2,2,2
2,Asha,1,60000,3,3,3
3,Vijay,1,50000,4,4,4
4,Neha,2,55000,1,1,1
5,Ravi,2,45000,2,2,2
6,Meena,3,70000,1,1,1
7,Kiran,3,50000,2,2,2


- `PARTITION BY department_id` restarts ranking for each department.
- `ROW_NUMBER()` assigns a unique sequential number to every row.
- `RANK()` gives tied values the same rank and leaves gaps after ties.
- `DENSE_RANK()` gives tied values the same rank without leaving gaps.


## Question 10: Filter Employees by Hire Date Using a CTE

**Interview prompt:**

Use a Common Table Expression (CTE) to find employees hired during 2023. Display their names in uppercase, their hire dates, and their years of service as of 2026.

**Concepts:** `WITH`, CTEs, `UPPER()`, date filtering, date functions.

In [18]:

query = """
WITH RecentHires AS (
    SELECT name, hire_date
    FROM Employees
    WHERE hire_date >= '2023-01-01'
      AND hire_date < '2024-01-01'
)
SELECT
    UPPER(name) AS employee_name,
    hire_date,
    2026 - CAST(SUBSTR(hire_date, 1, 4) AS INTEGER)
        AS years_since_hire_year
FROM RecentHires
ORDER BY hire_date;
"""

pd.read_sql_query(query, connection)


,employee_name,hire_date,years_since_hire_year
0,RAVI,2023-06-10,3
1,NEHA,2023-11-12,3


- A CTE is a named temporary result set defined using `WITH`.
- The CTE selects employees hired during 2023.
- `UPPER()` converts names to uppercase.
- `SUBSTR(hire_date, 1, 4)` extracts the year from the date string.
- Subtracting the hire year from 2026 gives the difference between calendar years, not the exact elapsed years and months.

**Note:** This example uses SQLite, which stores these dates as ISO-formatted text. Date functions differ between database systems.